# Step 8: Entanglement Entropy and the Area Law

All the observables we have computed so far — magnetisation, correlation functions, the spectral gap — have classical analogues. Entanglement does not. It is a purely quantum property: the ground state of a many-body system can be correlated in ways that no classical probability distribution can reproduce.

Here we compute the **entanglement entropy** of the TFIM ground state across its phase diagram. In the gapped phases it obeys the **area law** — a constant independent of subsystem size. At the critical point it grows logarithmically, with a coefficient that identifies the universality class through the **central charge** $c$ of the underlying conformal field theory.

This is also the theoretical foundation for tensor networks: the area law is precisely what makes matrix product states efficient.

**What you will do:**
- Build the reduced density matrix and perform the Schmidt decomposition via SVD
- Compute the von Neumann entanglement entropy from the Schmidt values
- Verify the area law in the ordered and disordered phases
- Fit the Calabrese-Cardy formula at criticality to extract the central charge $c = 1/2$
- Map the entanglement entropy across the full TFIM phase diagram
- Understand why the area law makes tensor networks possible

**Prerequisites:** Notebooks 06–07 (TFIM, ED, ground state calculation).

## 8.1  Bipartite Entanglement and the Reduced Density Matrix

Divide the $N$-site chain into subsystem $A$ (sites $0, \ldots, \ell-1$) and subsystem $B$ (sites $\ell, \ldots, N-1$). The two states of interest are:

- **Product state:** $|\psi\rangle = |\psi_A\rangle \otimes |\psi_B\rangle$. Subsystems are independent; measuring $A$ tells you nothing about $B$.
- **Entangled state:** $|\psi\rangle$ cannot be written as a product. The subsystems are quantum-mechanically correlated.

The **reduced density matrix** of subsystem $A$ is obtained by tracing out $B$:

$$\rho_A = \text{Tr}_B |E_0\rangle\langle E_0|$$

$\rho_A$ is a $2^\ell \times 2^\ell$ Hermitian matrix with non-negative eigenvalues summing to 1. It is a pure state ($\rho_A^2 = \rho_A$, rank 1) if and only if $A$ and $B$ are unentangled.

The **Schmidt decomposition** provides the most efficient route to $\rho_A$. Write the ground state vector as a matrix $C$ of shape $(2^\ell, 2^{N-\ell})$:

$$C_{ab} = \langle a_A, b_B | E_0 \rangle$$

and perform the SVD $C = U \Sigma V^\dagger$. The singular values $\lambda_\alpha$ are the **Schmidt coefficients**; the eigenvalues of $\rho_A$ are $\lambda_\alpha^2$.

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# ── TFIM Hamiltonian (from notebook 06) ─────────────────────────────────────
def build_tfim(N, J=1.0, Gamma=1.0, pbc=True):
    dim    = 2 ** N
    states = np.arange(dim)
    rows, cols, vals = [], [], []
    n_bonds = N if pbc else N - 1
    for i in range(n_bonds):
        j    = (i + 1) % N
        bi   = (states >> i) & 1
        bj   = (states >> j) & 1
        sz_i = 2 * bi.astype(float) - 1
        sz_j = 2 * bj.astype(float) - 1
        rows.extend(states.tolist())
        cols.extend(states.tolist())
        vals.extend((-J * sz_i * sz_j).tolist())
    for i in range(N):
        s_flip = states ^ (1 << i)
        rows.extend(states.tolist())
        cols.extend(s_flip.tolist())
        vals.extend([-Gamma] * dim)
    return csr_matrix((vals, (rows, cols)), shape=(dim, dim), dtype=float)


# ── Schmidt decomposition ────────────────────────────────────────────────────
def schmidt_values(psi0, N, ell):
    """
    Compute Schmidt coefficients for bipartition A = {0..ell-1}, B = {ell..N-1}.

    The coefficient tensor C[a, b] = psi0[a + 2^ell * b] is obtained by
    reshaping psi0 in Fortran (column-major) order: C = psi0.reshape(2^ell, 2^(N-ell), order='F').
    This follows from the bit encoding: a = s & (2^ell - 1),  b = s >> ell.

    Returns singular values lambda_alpha >= 0, sorted descending.
    """
    dim_A = 2 ** ell
    dim_B = 2 ** (N - ell)
    C     = psi0.reshape(dim_A, dim_B, order='F')
    return np.linalg.svd(C, compute_uv=False)   # singular values, sorted descending


# ── Verify on known two-site states ─────────────────────────────────────────
# Product state |up, up> = [0, 0, 0, 1] in (site0, site1) basis
# Encoding: s=3 = 11 in binary => both sites up => psi = [0,0,0,1]
psi_product = np.array([0.0, 0.0, 0.0, 1.0])
sv_product  = schmidt_values(psi_product, N=2, ell=1)
print("Product state |↑↑⟩:")
print(f"  Schmidt values: {sv_product}")
print(f"  (rank 1 → no entanglement)")

# Bell state (|↑↑⟩ + |↓↓⟩)/√2 = (psi[3] + psi[0])/√2
psi_bell = np.array([1.0, 0.0, 0.0, 1.0]) / np.sqrt(2)
sv_bell  = schmidt_values(psi_bell, N=2, ell=1)
print("\nBell state (|↑↑⟩ + |↓↓⟩)/√2:")
print(f"  Schmidt values: {sv_bell}")
print(f"  (rank 2 with equal weights → maximally entangled)")

# Reduced density matrix of Bell state: should be I/2
C_bell = psi_bell.reshape(2, 2, order='F')
rho_A  = C_bell @ C_bell.T
print(f"\n  rho_A = {rho_A}")
print(f"  (expected: [[0.5, 0], [0, 0.5]])")

## 8.2  Entanglement Entropy

The entanglement entropy is the von Neumann entropy of the reduced density matrix:

$$S_A = -\text{Tr}(\rho_A \log \rho_A) = -\sum_\alpha \lambda_\alpha^2 \log \lambda_\alpha^2$$

where the $\lambda_\alpha$ are the Schmidt values. The log is natural logarithm throughout.

Why this formula? The von Neumann entropy is the unique function of $\rho_A$ satisfying: (i) $S = 0$ for a pure state; (ii) $S$ is maximised for the maximally mixed state; (iii) $S$ is additive for independent subsystems. When evaluated on the Schmidt spectrum $\{\lambda_\alpha^2\}$, it reduces to the Shannon entropy of the probability distribution over Schmidt states.

Properties:
- $S_A = 0$: product state (single non-zero Schmidt value)
- $S_A = \log \chi$: maximally entangled state ($\chi$ equal Schmidt values $1/\sqrt{\chi}$)
- $S_A = S_B$: entanglement is symmetric (despite $\rho_A \neq \rho_B$ if subsystems differ in size)
- $S_A \leq \ell \log 2$: bounded by $\log$ of the Hilbert space dimension

In [ ]:
def entanglement_entropy(psi0, N, ell):
    """
    Von Neumann entanglement entropy for bipartition at site ell.
    S_A = -sum_alpha  lambda_alpha^2 * log(lambda_alpha^2)
    """
    sv   = schmidt_values(psi0, N, ell)
    sv2  = sv ** 2
    sv2  = sv2[sv2 > 1e-15]   # exclude numerically zero values to avoid log(0)
    return float(-np.sum(sv2 * np.log(sv2)))


# ── Verify on known states ───────────────────────────────────────────────────
print("Entanglement entropy checks:")
print(f"  Product state:          S = {entanglement_entropy(psi_product, 2, 1):.6f}  (expected 0)")
print(f"  Bell state:             S = {entanglement_entropy(psi_bell,    2, 1):.6f}  (expected {np.log(2):.6f} = log 2)")

# GHZ state (|↑↑↑⟩ + |↓↓↓⟩)/√2 for N=3
psi_ghz = np.zeros(8)
psi_ghz[0] = 1.0 / np.sqrt(2)   # |↓↓↓⟩  (s=0: all bits 0)
psi_ghz[7] = 1.0 / np.sqrt(2)   # |↑↑↑⟩  (s=7: all bits 1)
print(f"  GHZ state (ell=1):      S = {entanglement_entropy(psi_ghz, 3, 1):.6f}  (expected {np.log(2):.6f})")
print(f"  GHZ state (ell=2):      S = {entanglement_entropy(psi_ghz, 3, 2):.6f}  (expected {np.log(2):.6f})")

# Upper bound: ell*log(2) for ell=1
print(f"\n  Upper bound for ell=1:  S <= {1 * np.log(2):.6f} = log 2")
print(f"  Upper bound for ell=2:  S <= {2 * np.log(2):.6f} = 2 log 2")

# Symmetry: S_A = S_B
N_sym = 6
H_sym = build_tfim(N_sym, Gamma=0.8, pbc=True)
_, ev = eigsh(H_sym, k=1, which='SA')
psi_sym = ev[:, 0]
S_A = entanglement_entropy(psi_sym, N_sym, ell=2)
S_B = entanglement_entropy(psi_sym, N_sym, ell=N_sym - 2)
print(f"\n  S_A (ell=2) = {S_A:.8f}")
print(f"  S_B (ell=4) = {S_B:.8f}")
print(f"  S_A = S_B:  {np.isclose(S_A, S_B)}")

## 8.3  Area Law in the Gapped Phases

For a gapped ground state, correlations decay exponentially over the correlation length $\xi$. Spins far from the boundary between $A$ and $B$ are essentially unentangled with the other subsystem. The entanglement is therefore **local** — only the boundary region contributes — and $S_A$ saturates to a constant independent of $\ell$:

$$S_A \to S_{\max} = \text{const} \quad (\ell \gg \xi)$$

This is the **area law** in 1D (where the boundary consists of at most 2 points). In 2D it would say $S_A \propto L_{\partial A}$, the perimeter of the subsystem.

We test this at two points in the gapped TFIM:
- **Ordered phase:** $\Gamma/J = 0.3$ (strong ferromagnetic order)
- **Disordered phase:** $\Gamma/J = 2.0$ (strong transverse field)

In [ ]:
J      = 1.0
N_area = 24   # large enough to see saturation

Gamma_list  = [0.3 * J, 2.0 * J]
labels_list = [r'Ordered  ($\Gamma/J=0.3$)', r'Disordered  ($\Gamma/J=2.0$)']
color_list  = ['#2166ac', '#d6604d']

ell_values = np.arange(1, N_area // 2 + 1)
entropy_gapped = {}

for Gamma, label in zip(Gamma_list, labels_list):
    print(f"{label} ...", end=' ', flush=True)
    H       = build_tfim(N_area, J=J, Gamma=Gamma, pbc=True)
    _, evec = eigsh(H, k=1, which='SA')
    psi0    = evec[:, 0]
    S_arr   = [entanglement_entropy(psi0, N_area, ell) for ell in ell_values]
    entropy_gapped[label] = np.array(S_arr)
    print("done")

fig, ax = plt.subplots(figsize=(9, 5))
for label, col in zip(labels_list, color_list):
    ax.plot(ell_values, entropy_gapped[label], 'o-', ms=5, lw=1.8,
            color=col, label=label)

ax.set_xlabel(r'Subsystem size $\ell$', fontsize=13)
ax.set_ylabel(r'$S_A(\ell)$', fontsize=13)
ax.set_title(f'Entanglement entropy — gapped phases  ($N = {N_area}$)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

for label in labels_list:
    S = entropy_gapped[label]
    print(f"{label}:")
    print(f"  S(ell=1)   = {S[0]:.4f}")
    print(f"  S(ell=N/2) = {S[-1]:.4f}")
    print(f"  Variation  = {S[-1] - S[0]:.4f}  (area law: should be small)")

## 8.4  Logarithmic Scaling at the Critical Point

At $\Gamma_c = J$, the correlation length diverges and the area law is violated. For a 1D critical system described by a CFT, Calabrese and Cardy (2004) derived the exact scaling:

$$S_A(\ell) = \frac{c}{3} \log\!\left(\frac{N}{\pi}\sin\frac{\pi \ell}{N}\right) + k$$

for a chain of length $N$ with **periodic boundary conditions**, where $c$ is the **central charge** of the CFT and $k$ is a non-universal constant. For large $\ell \ll N$ this reduces to $(c/3)\log \ell + \text{const}$.

The TFIM critical point is described by the **Ising CFT** with central charge $c = 1/2$. We therefore expect:

$$S_A(\ell) = \frac{1}{6}\log\!\left(\frac{N}{\pi}\sin\frac{\pi \ell}{N}\right) + k$$

Fitting $c$ from this formula using ED data is one of the cleanest ways to identify the universality class of a quantum critical point — a single number extracted from a single measurement.

In [ ]:
# ── Entanglement entropy at criticality for several N ────────────────────────
N_crit_list = [16, 20, 24]
crit_colors = ['#4575b4', '#d73027', '#1a9850']

entropy_crit = {}
c_values     = []

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for N_c, col in zip(N_crit_list, crit_colors):
    print(f"N = {N_c} ...", end=' ', flush=True)
    H       = build_tfim(N_c, J=J, Gamma=J, pbc=True)
    _, evec = eigsh(H, k=1, which='SA')
    psi0    = evec[:, 0]

    ell_arr = np.arange(1, N_c // 2 + 1)
    S_arr   = np.array([entanglement_entropy(psi0, N_c, ell) for ell in ell_arr])
    entropy_crit[N_c] = (ell_arr, S_arr)

    # Calabrese-Cardy fit: S = (c/3) * log((N/pi)*sin(pi*ell/N)) + k
    x_cc = np.log((N_c / np.pi) * np.sin(np.pi * ell_arr / N_c))

    def cc_model(x, c_fit, k_fit):
        return (c_fit / 3.0) * x + k_fit

    popt, pcov = curve_fit(cc_model, x_cc, S_arr)
    c_fit, k_fit = popt
    c_values.append(c_fit)
    print(f"done  (c = {c_fit:.4f})")

    # Raw plot
    axes[0].plot(ell_arr, S_arr, 'o', ms=4, color=col)
    axes[0].plot(ell_arr, cc_model(x_cc, *popt), '-', lw=1.5, color=col,
                 label=f"$N={N_c}$, fit $c={c_fit:.3f}$")

    # Calabrese-Cardy collapse: S vs log((N/pi)*sin(pi*ell/N))
    axes[1].scatter(x_cc, S_arr, s=20, color=col, zorder=3)
    x_fine = np.linspace(x_cc.min(), x_cc.max(), 100)
    axes[1].plot(x_fine, cc_model(x_fine, *popt), '-', lw=1.5, color=col,
                 label=f"$N={N_c}$, $c={c_fit:.3f}$")

axes[0].set_xlabel(r'Subsystem size $\ell$', fontsize=13)
axes[0].set_ylabel(r'$S_A(\ell)$', fontsize=13)
axes[0].set_title(f'Entanglement at $\\Gamma = J$ (critical)', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel(r'$\log\!\left(\frac{N}{\pi}\sin\frac{\pi\ell}{N}\right)$', fontsize=13)
axes[1].set_ylabel(r'$S_A$', fontsize=13)
axes[1].set_title(r'Calabrese-Cardy fit: slope $= c/3$', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nCentral charge fits: {[f'{c:.4f}' for c in c_values]}")
print(f"Expected: c = 0.5000  (Ising CFT)")

## 8.5  Entanglement across the Phase Diagram

By varying $\Gamma/J$ and computing $S_A$ at the centre cut ($\ell = N/2$), we see the three-way distinction that characterises the TFIM phase diagram through entanglement alone:

- **Ordered phase** ($\Gamma < \Gamma_c$): area law, $S$ saturates to a value reflecting the two-fold degeneracy of the ground state.
- **Critical point** ($\Gamma = \Gamma_c$): $S$ is maximised and grows with $N$ as $\frac{c}{3}\log N$.
- **Disordered phase** ($\Gamma > \Gamma_c$): area law again, but to a smaller constant (the disordered ground state is closer to a product state).

This distinction — two area law phases separated by a critical point with enhanced entanglement — is a universal feature of 1D quantum systems and can be used as a diagnostic even when no order parameter is known.

In [ ]:
# S(ell=N/2) vs Gamma/J for several N
Gamma_scan = np.linspace(0.2, 2.0, 40)
N_scan     = [12, 16, 20, 24]
cols_scan  = plt.cm.cool(np.linspace(0.1, 0.9, len(N_scan)))

S_mid = {N: [] for N in N_scan}

for N in N_scan:
    print(f"N = {N} ...", end=' ', flush=True)
    for G in Gamma_scan:
        H       = build_tfim(N, J=J, Gamma=G, pbc=True)
        _, evec = eigsh(H, k=1, which='SA')
        S_mid[N].append(entanglement_entropy(evec[:, 0], N, N // 2))
    S_mid[N] = np.array(S_mid[N])
    print("done")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for N, col in zip(N_scan, cols_scan):
    axes[0].plot(Gamma_scan / J, S_mid[N], lw=2, color=col, label=f"$N = {N}$")
axes[0].axvline(1.0, color='k', ls='--', lw=1.2, label=r'$\Gamma_c = J$')
axes[0].set_xlabel(r'$\Gamma / J$', fontsize=13)
axes[0].set_ylabel(r'$S_A(\ell = N/2)$', fontsize=13)
axes[0].set_title('Half-chain entanglement entropy', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3)

# Critical point: S(N/2) ~ (c/3) log N + const
S_peak = np.array([S_mid[N][np.argmin(np.abs(Gamma_scan - J))] for N in N_scan])
slope_S, intercept_S = np.polyfit(np.log(N_scan), S_peak, 1)
N_fit = np.linspace(min(N_scan) * 0.9, max(N_scan) * 1.1, 50)

axes[1].semilogx(N_scan, S_peak, 'o', ms=8, color='#d73027', label='$S(\\ell=N/2)$ at $\\Gamma=J$')
axes[1].semilogx(N_fit, slope_S * np.log(N_fit) + intercept_S, 'k--', lw=1.5,
                 label=f'Fit slope = {slope_S:.3f}')
axes[1].semilogx(N_fit, (1/6) * np.log(N_fit) + intercept_S, 'r:', lw=1.5,
                 label=r'Expected: $c/3 = 1/6 \approx 0.167$')
axes[1].set_xlabel('System size $N$ (log scale)', fontsize=13)
axes[1].set_ylabel(r'$S_A(N/2)$', fontsize=13)
axes[1].set_title(r'Peak entropy grows as $(c/3)\log N$', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

print(f"Fitted slope: {slope_S:.4f}  (expected c/3 = {1/6:.4f}  for c=1/2)")

## 8.6  The Entanglement Spectrum

The full set of Schmidt values $\{\lambda_\alpha\}$, plotted as $-2\log\lambda_\alpha$ and sorted in ascending order, is the **entanglement spectrum**. It contains more information than the entanglement entropy alone.

For gapped topological phases, the structure of the entanglement spectrum encodes the edge modes of the system — this is Li and Haldane's (2008) conjecture that the entanglement spectrum of a topological phase resembles the physical spectrum of the system's edge. For the simple TFIM here we see a more modest signal: the number of significant Schmidt values (the effective Schmidt rank) changes qualitatively across the phase transition.

In [ ]:
N_spec = 20
ell_spec = N_spec // 2

Gamma_spec = [0.3 * J, J, 2.0 * J]
label_spec = [r'Ordered ($\Gamma/J=0.3$)', r'Critical ($\Gamma=J$)', r'Disordered ($\Gamma/J=2$)']
col_spec   = ['#2166ac', '#d6604d', '#1a9850']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for G, lbl, col in zip(Gamma_spec, label_spec, col_spec):
    H       = build_tfim(N_spec, J=J, Gamma=G, pbc=True)
    _, evec = eigsh(H, k=1, which='SA')
    psi0    = evec[:, 0]

    sv      = schmidt_values(psi0, N_spec, ell_spec)
    # Entanglement spectrum: xi_alpha = -2 log(lambda_alpha)
    sv_sig  = sv[sv > 1e-8]
    xi      = -2 * np.log(sv_sig)

    axes[0].plot(np.arange(1, len(xi) + 1), xi, 'o-', ms=4, lw=1.2,
                 color=col, label=lbl)

    # Schmidt value distribution
    axes[1].semilogy(np.arange(1, len(sv_sig) + 1), sv_sig ** 2, 'o-',
                     ms=4, lw=1.2, color=col, label=lbl)

axes[0].set_xlabel(r'Schmidt index $\alpha$', fontsize=13)
axes[0].set_ylabel(r'$\xi_\alpha = -2\log\lambda_\alpha$', fontsize=13)
axes[0].set_title('Entanglement spectrum', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].set_xlim(0, 30)
axes[0].grid(alpha=0.3)

axes[1].set_xlabel(r'Schmidt index $\alpha$', fontsize=13)
axes[1].set_ylabel(r'Schmidt weight $\lambda_\alpha^2$', fontsize=13)
axes[1].set_title('Schmidt weight distribution', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].set_xlim(0, 30)
axes[1].grid(alpha=0.3, which='both')

plt.tight_layout()
plt.show()

# Count effective Schmidt rank (weights > 1% of maximum)
print("Effective Schmidt rank (weights > 1% of max):")
for G, lbl in zip(Gamma_spec, label_spec):
    H       = build_tfim(N_spec, J=J, Gamma=G, pbc=True)
    _, evec = eigsh(H, k=1, which='SA')
    sv      = schmidt_values(evec[:, 0], N_spec, ell_spec)
    sv2     = sv ** 2
    rank    = int(np.sum(sv2 > 0.01 * sv2[0]))
    print(f"  {lbl}: chi_eff = {rank}")

## 8.7  From Entanglement to Tensor Networks

The entanglement entropy directly controls the cost of representing a quantum state numerically.

The Schmidt decomposition gives:

$$|E_0\rangle = \sum_{\alpha=1}^{\chi} \lambda_\alpha |\alpha\rangle_A |\alpha\rangle_B$$

where $\chi$ is the number of significant Schmidt values (those above some threshold $\epsilon$). The discarded weight is $\sum_{\alpha > \chi} \lambda_\alpha^2 \leq \epsilon^2$. For a state satisfying the area law, the Schmidt values decay exponentially: $\lambda_\alpha \sim e^{-\alpha/\xi_E}$ for some entanglement length $\xi_E$. A bond dimension $\chi \sim e^{S_{\max}}$ captures essentially all the weight — and $S_{\max}$ is bounded by a constant independent of system size.

This is why **Matrix Product States** work. An MPS with bond dimension $\chi$ requires storing $O(N \chi^2)$ numbers — linear in system size. For a gapped state with constant $S_{\max}$, a fixed $\chi$ gives exponentially accurate results regardless of $N$.

At a critical point, $S_A \sim (c/3)\log\ell$, so the Schmidt rank grows as a power law: $\chi \sim \ell^{c/6}$. This is still far better than the exact $2^{N/2}$ that would be needed for a generic state, which is why DMRG (notebook 10) can study critical systems efficiently, just with a larger bond dimension.

| Phase | $S_A(\ell)$ | Required $\chi$ | MPS cost |
|---|---|---|---|
| Gapped | Const | Const | $O(N)$ |
| Critical | $(c/3)\log\ell$ | $\ell^{c/6}$ | Polynomial in $N$ |
| Generic | $O(\ell)$ | $e^{O(\ell)}$ | Exponential in $N$ |

In notebook 09 we construct MPS by hand using iterative SVD truncation, and in notebook 10 we use DMRG (TeNPy) to study systems far beyond the reach of exact diagonalisation.

In [ ]:
# Visualise the Schmidt value decay in the three phases
# and the truncation error as a function of bond dimension chi

N_trunc  = 20
ell_trunc = N_trunc // 2
chi_range = np.arange(1, 50)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for G, lbl, col in zip(Gamma_spec, label_spec, col_spec):
    H       = build_tfim(N_trunc, J=J, Gamma=G, pbc=True)
    _, evec = eigsh(H, k=1, which='SA')
    psi0    = evec[:, 0]
    sv      = schmidt_values(psi0, N_trunc, ell_trunc)
    sv2     = sv ** 2

    # Truncation error: sum of discarded weights
    trunc_err = np.array([
        float(np.sum(sv2[chi:])) if chi < len(sv2) else 0.0
        for chi in chi_range
    ])

    axes[0].semilogy(np.arange(1, len(sv2) + 1), sv2, 'o-', ms=3,
                     lw=1.2, color=col, label=lbl)
    axes[1].semilogy(chi_range, np.maximum(trunc_err, 1e-16), '-',
                     lw=2, color=col, label=lbl)

axes[0].set_xlabel(r'Schmidt index $\alpha$', fontsize=13)
axes[0].set_ylabel(r'$\lambda_\alpha^2$', fontsize=13)
axes[0].set_title('Schmidt weight decay', fontsize=13)
axes[0].legend(fontsize=9)
axes[0].grid(alpha=0.3, which='both')

axes[1].axhline(1e-4, color='grey', ls=':', lw=1.2, label='$10^{-4}$ threshold')
axes[1].set_xlabel(r'Bond dimension $\chi$', fontsize=13)
axes[1].set_ylabel('Truncation error', fontsize=13)
axes[1].set_title('Error vs bond dimension', fontsize=13)
axes[1].legend(fontsize=9)
axes[1].grid(alpha=0.3, which='both')

plt.suptitle(f'MPS approximation quality  ($N={N_trunc}$, $\\ell=N/2$)', fontsize=13)
plt.tight_layout()
plt.show()

# Chi needed to reach truncation error < 1e-4
print("Bond dimension chi needed for truncation error < 1e-4:")
for G, lbl in zip(Gamma_spec, label_spec):
    H       = build_tfim(N_trunc, J=J, Gamma=G, pbc=True)
    _, evec = eigsh(H, k=1, which='SA')
    sv2     = schmidt_values(evec[:, 0], N_trunc, ell_trunc) ** 2
    trunc   = np.array([float(np.sum(sv2[chi:])) if chi < len(sv2) else 0.0
                        for chi in chi_range])
    below   = np.where(trunc < 1e-4)[0]
    chi_min = int(chi_range[below[0]]) if len(below) > 0 else '>50'
    print(f"  {lbl}: chi = {chi_min}")

## Exercises

1. **Entanglement entropy of the Heisenberg chain.** The 1D antiferromagnetic Heisenberg chain is also critical (gapless, $c = 1$). Using `build_heisenberg` from notebook 05 with PBC, compute $S_A(\ell)$ at the midpoint cut and fit the Calabrese-Cardy formula to extract the central charge. You should find $c \approx 1$.

2. **OBC correction.** For open boundary conditions, the Calabrese-Cardy formula is modified to $S_A(\ell) = \frac{c}{6}\log\!\left(\frac{2N}{\pi}\sin\frac{\pi\ell}{N}\right) + k$, with a factor of $c/6$ instead of $c/3$. Repeat the central charge extraction for the TFIM with OBC and verify you still recover $c = 1/2$.

3. **Entanglement entropy vs correlation length.** In the gapped phases, the area law constant $S_{\max}$ grows as the correlation length $\xi$ increases. Near the critical point, $\xi \sim |\Gamma - \Gamma_c|^{-\nu}$ with $\nu = 1$. Plot $S_{\max}(\ell = N/2)$ vs $|\Gamma - \Gamma_c|^{-1}$ for $\Gamma > \Gamma_c$ (several $N$) and check whether the two scales track each other.

4. **Rényi entropy.** The $n$-th Rényi entropy is $S_n = \frac{1}{1-n}\log\text{Tr}(\rho_A^n) = \frac{1}{1-n}\log\sum_\alpha \lambda_\alpha^{2n}$. Compute $S_2$ (the most experimentally accessible) at the critical point for $N = 20$ and fit the Calabrese-Cardy form with the known modification $c \to c(1 + 1/n)/2$. Verify that $c$ is consistent with $1/2$.